# Method 1 - Part 3: TESSERA and AlphaEarth fusion with the light UNet

Evaluates whether fusing TESSERA 2019 (128 ch.) and AlphaEarth 2019 (64 ch.) improves crop segmentation over each single embedding when trained with the same `32-64-128` UNet architecture, 3,600-step budget, and weighted cross-entropy loss on fold I (100% labels, seeds 0, 1, 2).

| Variant | Description |
|---|---|
| `base_unet-t19-w32` | TESSERA 2019 alone (128 ch., from Baselines Part 3) |
| `base_unet-ae19-w32` | AlphaEarth 2019 alone (64 ch.) |
| `m1_unet-t19-ae19-w32` | Early concatenation of TESSERA 2019 + AlphaEarth 2019 (192 ch.) |
| `m1_unet-t19-noise-w32` | TESSERA 2019 + 64 N(0, 1) noise channels (dimensionality control) |
| `m1_gated-t19-ae19-w32` | Pixel-wise gated fusion `a * f1(T) + (1 - a) * f2(AE)` before the UNet |
| `m1_late-t19-ae19` | Late fusion (mean logits of the trained TESSERA and AlphaEarth UNets) |

## 1. Environment and paths

In [ ]:
import json, os, shutil, platform, sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
print("Python", platform.python_version(), "| torch", torch.__version__, f"| {os.cpu_count()} CPUs")
assert torch.cuda.is_available(), "no GPU: Runtime > Change runtime type > T4 GPU, then run again"
print("GPU:", torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

from google.colab import drive
drive.mount("/content/drive")

DRIVE = Path("/content/drive/MyDrive/pj_vmi")
REPO_URL = "https://github.com/ColdCastlee/ProjectVMI.git"
BRANCH = "main"  # or the branch being tested
CODE = Path("/content/code")
PROC_DRIVE = DRIVE / "processed"
PROC = Path("/content/processed")
RUNS, RUNS_CSV = DRIVE / "runs", DRIVE / "results" / "runs.csv"
OUT = DRIVE / "results" / "m1"
SEEDS = "0 1 2"
B5 = "configs/baselines/b5_unet_ae19.yaml"
M1_1 = "configs/m1/m1_1_concat_t19_ae19.yaml"
M1_2 = "configs/m1/m1_2_noise_t19.yaml"
M1_4 = "configs/m1/m1_4_gated_t19_ae19.yaml"

missing = [s for s in range(3) if not (RUNS / f"base_unet-t19-w32_100_foldI_s{s}" / "run.json").exists()]
assert not missing, f"B2 seeds {missing} missing: run notebooks/baselines/03_b2.ipynb first"
print("B2 runs found for seeds 0, 1, 2")

## 2. Repository and unit tests

In [ ]:
if (CODE / ".git").exists():
    !git -C "{CODE}" fetch -q origin {BRANCH} && git -C "{CODE}" checkout -q {BRANCH} && git -C "{CODE}" pull -q --ff-only origin {BRANCH}
else:
    !git clone -q -b {BRANCH} {REPO_URL} "{CODE}"
%cd {CODE}
!git log -1 --format="commit %h  %ad  %s" --date=short
!pip install -q -r requirements.txt
!python -m pytest -q tests
sys.path.insert(0, str(CODE))

## 3. Copy TESSERA 2019 and AlphaEarth 2019 to local disk

In [ ]:
PROC.mkdir(parents=True, exist_ok=True)
for f in ["meta.csv", "splits.json", "labels.npy", "instances.npy", "valid_mask.npy",
          "tessera_2019_q.npy", "tessera_2019_scale.npy", "ae_2019.npy"]:
    if not (PROC / f).exists() or (PROC / f).stat().st_size != (PROC_DRIVE / f).stat().st_size:
        shutil.copy(PROC_DRIVE / f, PROC / f)
    print(f"{f:24s} {(PROC / f).stat().st_size / 1e9:6.2f} GB  ok={(PROC / f).stat().st_size == (PROC_DRIVE / f).stat().st_size}")

## 4. AlphaEarth 2019 baseline (`base_unet-ae19-w32`)
Train the `32-64-128` UNet on AlphaEarth 2019 alone (64 input channels) across seeds 0, 1, and 2.

In [ ]:
!python scripts/train_head.py --config {B5} --processed {PROC} --runs-dir {RUNS} --runs-csv {RUNS_CSV} --seeds {SEEDS}

## 5. Early concatenation (`m1_unet-t19-ae19-w32`)
Stack TESSERA 2019 and AlphaEarth 2019 along the channel dimension (192 channels total) and train the UNet across seeds 0, 1, and 2.

In [ ]:
!python scripts/train_head.py --config {M1_1} --processed {PROC} --runs-dir {RUNS} --runs-csv {RUNS_CSV} --seeds {SEEDS}

## 6. Dimensionality control (`m1_unet-t19-noise-w32`)
Replace the 64 AlphaEarth channels with deterministic `N(0, 1)` Gaussian noise per `(seed, patch)` to test whether any gain from concatenation comes merely from wider input projections.

In [ ]:
!python scripts/train_head.py --config {M1_2} --processed {PROC} --runs-dir {RUNS} --runs-csv {RUNS_CSV} --seeds {SEEDS}

## 7. Gated fusion (`m1_gated-t19-ae19-w32`)
Project each embedding to 32 channels with a `1x1` convolution and blend them per pixel using a learned sigmoid gate `a = sigmoid(g([T; AE]))` before passing the result into the UNet.

In [ ]:
!python scripts/train_head.py --config {M1_4} --processed {PROC} --runs-dir {RUNS} --runs-csv {RUNS_CSV} --seeds {SEEDS}

## 8. Late fusion (`m1_late-t19-ae19`)
Average the predicted logits of the trained `base_unet-t19-w32` and `base_unet-ae19-w32` models for each seed without additional training.

In [ ]:
!python scripts/m1_late_fusion.py --processed {PROC} --runs-dir {RUNS} --runs-csv {RUNS_CSV} --a base_unet-t19-w32 --b base_unet-ae19-w32 --variant late-t19-ae19 --seeds {SEEDS}

## 9. Test set summary and paired bootstrap comparisons
Compare all single-source and fusion variants across seeds 0, 1, and 2 using patch-bootstrap confidence intervals and paired differences against each single embedding, the noise control, and early concatenation.

In [ ]:
from IPython.display import Image, display
from scripts.summarize_runs import summarize_runs

T, AE = "base_unet-t19-w32", "base_unet-ae19-w32"
CONCAT, NOISE, GATED, LATE = "m1_unet-t19-ae19-w32", "m1_unet-t19-noise-w32", "m1_gated-t19-ae19-w32", "m1_late-t19-ae19"
pairs = [f"{v}:{ref}" for v in (CONCAT, GATED, LATE) for ref in (T, AE)]
pairs += [f"{NOISE}:{T}", f"{CONCAT}:{NOISE}", f"{GATED}:{CONCAT}", f"{LATE}:{CONCAT}"]
pd.set_option("display.width", 220)
per_seed, summary, deltas = summarize_runs(RUNS, OUT, name="fusion_unet_foldI", variants=[T, AE, CONCAT, NOISE, GATED, LATE],
                                           pairs=pairs, split="I", label_frac=1.0, seeds=(0, 1, 2))
print(summary[["variant", "seeds", "miou_mean", "miou_std", "ci_low", "ci_high", "boundary_mean", "core_mean",
               "n_params", "train_seconds_mean"]].round(4).to_string(index=False))
print("\npaired bootstrap, delta = mIoU(variant) - mIoU(versus), averaged over seeds (all valid pixels):")
print(deltas[deltas.region == "all"].drop(columns="region").round(4).to_string(index=False))
display(Image(str(OUT / "fusion_unet_foldI_per_class.png")))